# M2 Experiments: Heuristic Product Retrieval

Colab-ready starter notebook for **CS 57200 – Heuristic Problem Solving (Track B)**.

This notebook demonstrates:
- correct baseline A*
- enhancement 1 (tight heuristic)
- brute-force correctness checks on tiny instances
- baseline vs enhancement experiments
- scaling plots


## 1. Setup

If you opened this notebook directly from GitHub, it is usually **read-only** in Colab.
Click **Copy to Drive** at the top first if you want to edit it.

If the repo is not already present in `/content`, clone it manually in the next cell.


In [ ]:
# Uncomment this if the repo is not already cloned in Colab
# !git clone https://github.com/sowjanyakaranam/heuristic-product-retrieval.git


In [ ]:
import os
import sys

ROOT = "/content/heuristic-product-retrieval"

if os.path.exists(ROOT):
    os.chdir(ROOT)

sys.path.append(os.path.join(os.getcwd(), "src"))

print("Working directory:", os.getcwd())
print("Repo exists:", os.path.exists(ROOT))
print("Top-level files:", os.listdir(os.getcwd())[:10])


In [ ]:
!pip install -r requirements.txt


## 2. Imports


In [ ]:
from data_utils import load_products_csv, filter_by_category
from experiments import run_correctness_test, run_baseline_vs_tight, run_scaling_experiment
from viz import plot_algorithm_comparison, plot_scaling
import pandas as pd


## 3. Load Sample Data


In [ ]:
products = load_products_csv("data/sample_products.csv")
snacks = filter_by_category(products, "Snacks")
query_embedding = (0.22, 0.32, 0.44, 0.25)

print("Total products:", len(products))
print("Snack products:", len(snacks))
snacks[0]


## 4. Correctness Tests

These compare A* with brute force on small instances.


In [ ]:
r1 = run_correctness_test(snacks[:8], query_embedding, k=3, lam=0.30)
r2 = run_correctness_test(snacks[:10], query_embedding, k=4, lam=0.30)

bev = filter_by_category(products, "Beverages")[:8]
q_bev = (0.45, 0.34, 0.61, 0.29)
r3 = run_correctness_test(bev, q_bev, k=3, lam=0.30)

df_correctness = pd.DataFrame([r1, r2, r3])
df_correctness


## 5. Experiment 1: Baseline vs Tight Heuristic


In [ ]:
exp1 = [run_baseline_vs_tight(snacks[:12], query_embedding, k=k, lam=0.30) for k in [3, 4, 5]]
df_exp1 = pd.DataFrame(exp1)
df_exp1


In [ ]:
plot_algorithm_comparison(
    labels=[str(x) for x in df_exp1["k"]],
    baseline_vals=df_exp1["baseline_nodes"].tolist(),
    tight_vals=df_exp1["tight_nodes"].tolist(),
    ylabel="Nodes Expanded",
    title="Baseline A* vs Tight Heuristic: Nodes Expanded",
    output_path="results/figures/baseline_vs_tight_nodes.png",
)


## 6. Experiment 2: Scaling Study


In [ ]:
pool = products[:40]
scaling = run_scaling_experiment(pool, query_embedding, sizes=[10, 15, 20, 25], k=4, lam=0.30)
df_scaling = pd.DataFrame(scaling)
df_scaling


In [ ]:
plot_scaling(
    ns=df_scaling["n"].tolist(),
    baseline_vals=df_scaling["baseline_runtime_sec"].tolist(),
    tight_vals=df_scaling["tight_runtime_sec"].tolist(),
    ylabel="Runtime (sec)",
    title="Scaling Study: Runtime vs Candidate Pool Size",
    output_path="results/figures/scaling_runtime.png",
)


## 7. Save Tables for the Report


In [ ]:
df_correctness.to_csv("results/tables/correctness_tests.csv", index=False)
df_exp1.to_csv("results/tables/exp1_baseline_vs_tight.csv", index=False)
df_scaling.to_csv("results/tables/exp2_scaling.csv", index=False)
print("Saved tables and figures under results/.")


## 8. Next Steps

- Replace `data/sample_products.csv` with your cleaned real category data
- Replace synthetic embeddings with Sentence-BERT embeddings
- Add a third experiment varying `K` or `lambda`
- Save results and use them directly in your M2 report
